In [125]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns  # 导入Seaborn库，用于统计数据可视化
plt.rcParams['font.sans-serif'] = ['SimHei']

1. 数据加载

In [126]:
train_data = pd.read_csv("./health_lifestyle_classification.csv",encoding='utf-8')

In [90]:
train_data

,survey_code,age,gender,height,weight,bmi,bmi_estimated,bmi_scaled,bmi_corrected,waist_size,...,sunlight_exposure,meals_per_day,caffeine_intake,family_history,pet_owner,electrolyte_level,gene_marker_flag,environmental_risk_score,daily_supplement_dosage,target
0,1,56,Male,173.416872,56.886640,18.915925,18.915925,56.747776,18.989117,72.165130,...,High,5,Moderate,No,Yes,0,1.0,5.5,-2.275502,healthy
1,2,69,Female,163.207380,97.799859,36.716278,36.716278,110.148833,36.511417,85.598889,...,High,5,High,Yes,No,0,1.0,5.5,6.239340,healthy
2,3,46,Male,177.281966,80.687562,25.673050,25.673050,77.019151,25.587429,90.295030,...,High,4,Moderate,No,No,0,1.0,5.5,5.423737,healthy
3,4,32,Female,172.101255,63.142868,21.318480,21.318480,63.955440,21.177109,100.504211,...,High,1,None,No,Yes,0,1.0,5.5,8.388611,healthy
4,5,60,Female,163.608816,40.000000,14.943302,14.943302,44.829907,14.844299,69.021150,...,High,1,High,Yes,Yes,0,1.0,5.5,0.332622,healthy
5,6,25,Male,186.788025,55.276111,15.843073,15.843073,47.529218,16.087835,86.591923,...,High,4,None,Yes,Yes,0,1.0,5.5,-8.985465,healthy
6,7,78,Female,166.164733,65.035212,23.554335,23.554335,70.663006,23.452755,67.838374,...,High,2,None,No,No,0,NaN,5.5,-6.296536,healthy
7,8,38,Female,163.783995,47.756125,17.802712,17.802712,53.408136,17.654781,73.071574,...,Low,3,Moderate,No,Yes,0,1.0,5.5,7.875266,healthy
8,9,56,Female,165.716460,47.257109,17.208216,17.208216,51.624648,17.199712,46.675208,...,High,2,Moderate,No,No,0,NaN,5.5,-8.055654,healthy
9,10,75,Male,151.310888,65.966591,28.812682,28.812682,86.438045,29.088049,94.294368,...,High,1,High,Yes,No,0,1.0,5.5,-3.035162,healthy


In [127]:
# 分离目标变量(方法1)
y = train_data.pop('target')   # 删除列名指定的列,同时将结果返回到y_train

In [128]:
# 初步检查
# print(f"数据维度: {train_df.shape}")
print("缺失值统计:\n", train_data.isnull().sum().sort_values(ascending=False).head(10))

缺失值统计:
 insulin                15836
heart_rate             14003
alcohol_consumption    13910
gene_marker_flag       10474
income                  8470
daily_steps             8329
blood_pressure          7669
bmi                        0
bmi_estimated              0
calorie_intake             0
dtype: int64


In [129]:
# 通过set_index()将第一列ID设为索引列，规范数据格式
train_df = train_data.set_index(train_data.columns[0])  # set_index设置新的索引，并删除该列

In [115]:
train_df

,age,gender,height,weight,bmi,bmi_estimated,bmi_scaled,bmi_corrected,waist_size,blood_pressure,...,insurance,sunlight_exposure,meals_per_day,caffeine_intake,family_history,pet_owner,electrolyte_level,gene_marker_flag,environmental_risk_score,daily_supplement_dosage
survey_code,,,,,,,,,,,,,,,,,,,,,
1,56,Male,173.416872,56.886640,18.915925,18.915925,56.747776,18.989117,72.165130,118.264254,...,No,High,5,Moderate,No,Yes,0,1.0,5.5,-2.275502
2,69,Female,163.207380,97.799859,36.716278,36.716278,110.148833,36.511417,85.598889,117.917986,...,No,High,5,High,Yes,No,0,1.0,5.5,6.239340
3,46,Male,177.281966,80.687562,25.673050,25.673050,77.019151,25.587429,90.295030,123.073698,...,Yes,High,4,Moderate,No,No,0,1.0,5.5,5.423737
4,32,Female,172.101255,63.142868,21.318480,21.318480,63.955440,21.177109,100.504211,148.173453,...,No,High,1,None,No,Yes,0,1.0,5.5,8.388611
5,60,Female,163.608816,40.000000,14.943302,14.943302,44.829907,14.844299,69.021150,150.613181,...,Yes,High,1,High,Yes,Yes,0,1.0,5.5,0.332622
6,25,Male,186.788025,55.276111,15.843073,15.843073,47.529218,16.087835,86.591923,118.722971,...,Yes,High,4,None,Yes,Yes,0,1.0,5.5,-8.985465
7,78,Female,166.164733,65.035212,23.554335,23.554335,70.663006,23.452755,67.838374,102.141757,...,Yes,High,2,None,No,No,0,NaN,5.5,-6.296536
8,38,Female,163.783995,47.756125,17.802712,17.802712,53.408136,17.654781,73.071574,136.790243,...,Yes,Low,3,Moderate,No,Yes,0,1.0,5.5,7.875266
9,56,Female,165.716460,47.257109,17.208216,17.208216,51.624648,17.199712,46.675208,133.536300,...,Yes,High,2,Moderate,No,No,0,NaN,5.5,-8.055654


## 一、数据预处理

### 1. 特征类型识别与处理

In [130]:
# 识别数值型特征
numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
print(f"数值型特征数量: {len(numeric_cols)}")

数值型特征数量: 29


In [66]:
print(numeric_cols)

['age', 'height', 'weight', 'bmi', 'bmi_estimated', 'bmi_scaled', 'bmi_corrected', 'waist_size', 'blood_pressure', 'heart_rate', 'cholesterol', 'glucose', 'insulin', 'sleep_hours', 'work_hours', 'physical_activity', 'daily_steps', 'calorie_intake', 'sugar_intake', 'water_intake', 'screen_time', 'stress_level', 'mental_health_score', 'income', 'meals_per_day', 'electrolyte_level', 'gene_marker_flag', 'environmental_risk_score', 'daily_supplement_dosage']


In [131]:
# 识别潜在的分类特征（唯一值较少的数值列）
potential_categorical = []
true_numeric_cols = []  # 真正的连续数值特征

for col in numeric_cols:
    if train_df[col].nunique() <= 11 and train_df[col].nunique() > 1:
        potential_categorical.append(col)
    else:
        true_numeric_cols.append(col)
        
print(f"潜在数值型分类特征: {len(potential_categorical)}")
print(f"真正的连续数值特征: {len(true_numeric_cols)}")

潜在数值型分类特征: 3
真正的连续数值特征: 26


In [132]:
print(potential_categorical)

['stress_level', 'mental_health_score', 'meals_per_day']


In [133]:
# 识别文本型特征
text_cols = train_df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"文本型特征数量: {len(text_cols)}")

文本型特征数量: 17


### 3.数据转换
将数值类型列保留两位小数

In [134]:
train_df[true_numeric_cols] = train_df[true_numeric_cols].astype(float).round(2)

In [135]:
train_df

,age,gender,height,weight,bmi,bmi_estimated,bmi_scaled,bmi_corrected,waist_size,blood_pressure,...,insurance,sunlight_exposure,meals_per_day,caffeine_intake,family_history,pet_owner,electrolyte_level,gene_marker_flag,environmental_risk_score,daily_supplement_dosage
survey_code,,,,,,,,,,,,,,,,,,,,,
1,56.0,Male,173.42,56.89,18.92,18.92,56.75,18.99,72.17,118.26,...,No,High,5,Moderate,No,Yes,0.0,1.0,5.5,-2.28
2,69.0,Female,163.21,97.80,36.72,36.72,110.15,36.51,85.60,117.92,...,No,High,5,High,Yes,No,0.0,1.0,5.5,6.24
3,46.0,Male,177.28,80.69,25.67,25.67,77.02,25.59,90.30,123.07,...,Yes,High,4,Moderate,No,No,0.0,1.0,5.5,5.42
4,32.0,Female,172.10,63.14,21.32,21.32,63.96,21.18,100.50,148.17,...,No,High,1,None,No,Yes,0.0,1.0,5.5,8.39
5,60.0,Female,163.61,40.00,14.94,14.94,44.83,14.84,69.02,150.61,...,Yes,High,1,High,Yes,Yes,0.0,1.0,5.5,0.33
6,25.0,Male,186.79,55.28,15.84,15.84,47.53,16.09,86.59,118.72,...,Yes,High,4,None,Yes,Yes,0.0,1.0,5.5,-8.99
7,78.0,Female,166.16,65.04,23.55,23.55,70.66,23.45,67.84,102.14,...,Yes,High,2,None,No,No,0.0,NaN,5.5,-6.30
8,38.0,Female,163.78,47.76,17.80,17.80,53.41,17.65,73.07,136.79,...,Yes,Low,3,Moderate,No,Yes,0.0,1.0,5.5,7.88
9,56.0,Female,165.72,47.26,17.21,17.21,51.62,17.20,46.68,133.54,...,Yes,High,2,Moderate,No,No,0.0,NaN,5.5,-8.06


### 2. 缺失值处理

In [136]:
# 真正的连续数值特征用中位数填充
if true_numeric_cols:
    train_df[true_numeric_cols] = train_df[true_numeric_cols].fillna(train_df[true_numeric_cols].median())
    print("连续数值特征缺失值已用中位数填充")

连续数值特征缺失值已用中位数填充


In [137]:
# 数值型分类特征用众数填充
if potential_categorical:
    for col in potential_categorical:
        if train_df[col].notna().any():
            # 对于数值型分类，用众数填充
            mode_val = train_df[col].mode()[0] if not train_df[col].mode().empty else train_df[col].median()
            train_df[col] = train_df[col].fillna(mode_val)
    print("数值型分类特征缺失值已用众数填充")

数值型分类特征缺失值已用众数填充


In [138]:
# 文本型特征用众数填充
if text_cols:
    for col in text_cols:
        if train_df[col].notna().any():
            mode_val = train_df[col].mode()[0] if not train_df[col].mode().empty else "MISSING"
            train_df[col] = train_df[col].fillna(mode_val)
    print("文本型特征缺失值已用众数填充")

文本型特征缺失值已用众数填充


In [122]:
train_df.shape

(100000, 46)

In [139]:
# 检查是否填充完整
print("缺失值统计:\n", train_df.isnull().sum().sort_values(ascending=False).head(10))

缺失值统计:
 daily_supplement_dosage    0
cholesterol                0
sugar_intake               0
calorie_intake             0
daily_steps                0
physical_activity          0
work_hours                 0
sleep_quality              0
sleep_hours                0
insulin                    0
dtype: int64


### 4. 特征编码

In [140]:
# 文本特征进行One-Hot编码
if text_cols:
    train_df_new = pd.get_dummies(train_df, columns=text_cols, prefix_sep="::")
    print(f"文本特征One-Hot编码后新增 {len(train_df_new.columns) - len(train_df.columns)} 列")

文本特征One-Hot编码后新增 39 列


### 4. 特征选择

In [34]:
# # 移除低方差特征
# from sklearn.feature_selection import VarianceThreshold

# selector = VarianceThreshold(threshold=0.1)
# X_selected = selector.fit_transform(train_df_new)   # 移除低方差特征后的数据 ---> array（数组）

In [141]:
# 移除低方差特征
from sklearn.feature_selection import VarianceThreshold

# 移除低方差特征（阈值0.1）
selector = VarianceThreshold(threshold=0.1)
X_selected = selector.fit_transform(train_df_new)

# 重建DataFrame
selected_columns = train_df_new.columns[selector.get_support()]
train_df_selected = pd.DataFrame(X_selected, columns=selected_columns)

print("\n移除低方差特征后的DataFrame:")
print(train_df_selected.shape)


移除低方差特征后的DataFrame:
(100000, 82)


In [142]:
print(f"特征选择前: {train_df_new.shape[1]} 个特征")
print(f"特征选择后: {train_df_selected.shape[1]} 个特征")
print(f"移除了 {train_df_new.shape[1] - train_df_selected.shape[1]} 个低方差特征")

特征选择前: 85 个特征
特征选择后: 82 个特征
移除了 3 个低方差特征


In [143]:
type(train_df_selected)

pandas.core.frame.DataFrame

In [144]:
train_df_selected

,age,height,weight,bmi,bmi_estimated,bmi_scaled,bmi_corrected,waist_size,blood_pressure,heart_rate,...,sunlight_exposure::High,sunlight_exposure::Low,sunlight_exposure::Moderate,caffeine_intake::High,caffeine_intake::Moderate,caffeine_intake::None,family_history::No,family_history::Yes,pet_owner::No,pet_owner::Yes
0,56.0,173.42,56.89,18.92,18.92,56.75,18.99,72.17,118.26,60.75,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
1,69.0,163.21,97.80,36.72,36.72,110.15,36.51,85.60,117.92,66.46,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
2,46.0,177.28,80.69,25.67,25.67,77.02,25.59,90.30,123.07,76.04,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
3,32.0,172.10,63.14,21.32,21.32,63.96,21.18,100.50,148.17,68.78,...,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0
4,60.0,163.61,40.00,14.94,14.94,44.83,14.84,69.02,150.61,92.34,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
5,25.0,186.79,55.28,15.84,15.84,47.53,16.09,86.59,118.72,51.97,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
6,78.0,166.16,65.04,23.55,23.55,70.66,23.45,67.84,102.14,70.47,...,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0
7,38.0,163.78,47.76,17.80,17.80,53.41,17.65,73.07,136.79,65.79,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
8,56.0,165.72,47.26,17.21,17.21,51.62,17.20,46.68,133.54,78.21,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
9,75.0,151.31,65.97,28.81,28.81,86.44,29.09,94.29,121.76,90.47,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0


In [145]:
# 获取保留的特征名称
selected_features = train_df_new.columns[selector.get_support()].tolist()
selected_features

['age',
 'height',
 'weight',
 'bmi',
 'bmi_estimated',
 'bmi_scaled',
 'bmi_corrected',
 'waist_size',
 'blood_pressure',
 'heart_rate',
 'cholesterol',
 'glucose',
 'insulin',
 'sleep_hours',
 'work_hours',
 'physical_activity',
 'daily_steps',
 'calorie_intake',
 'sugar_intake',
 'water_intake',
 'screen_time',
 'stress_level',
 'mental_health_score',
 'income',
 'meals_per_day',
 'daily_supplement_dosage',
 'gender::Female',
 'gender::Male',
 'sleep_quality::Excellent',
 'sleep_quality::Fair',
 'sleep_quality::Good',
 'sleep_quality::Poor',
 'alcohol_consumption::None',
 'alcohol_consumption::Occasionally',
 'alcohol_consumption::Regularly',
 'smoking_level::Heavy',
 'smoking_level::Light',
 'smoking_level::Non-smoker',
 'mental_health_support::No',
 'mental_health_support::Yes',
 'education_level::Bachelor',
 'education_level::High School',
 'education_level::Master',
 'education_level::PhD',
 'job_type::Healthcare',
 'job_type::Labor',
 'job_type::Office',
 'job_type::Service',
 

In [146]:
len(selected_features)

82

## 当前数据集：

In [147]:
# 特征：train_df_selected
# 标签：y